Step 1: Upload our project code.
Here we upload our project zip file (which has all our Python files) and unzip it inside Colab so we can run our code.

In [1]:
from google.colab import files
print("Upload network-traffic-transformer-lstm.zip")
uploaded = files.upload()

import zipfile
with zipfile.ZipFile("network-traffic-transformer-lstm.zip", 'r') as zip_ref:
    zip_ref.extractall(".")

%cd network-traffic-transformer-lstm

Upload network-traffic-transformer-lstm.zip


Saving network-traffic-transformer-lstm.zip to network-traffic-transformer-lstm.zip
/content/network-traffic-transformer-lstm


Step 2: Install the extra libraries we need
Colab already has TensorFlow, pandas, numpy and scikit-learn installed, so we only install the few extra ones our project needs: seaborn (for plotting), joblib (for saving models), and pyarrow (for reading one of our dataset files, since it's in .parquet format).

In [2]:
!pip install -q seaborn joblib pyarrow

Step 3: Upload our first dataset (CIC-Darknet2020)
This is the same dataset the base paper used. We downloaded it from Kaggle and upload it here.

In [3]:
import os
os.makedirs('/content/network-traffic-transformer-lstm/dataset/raw', exist_ok=True)

from google.colab import files
print("Upload cicdarknet2020.parquet")
uploaded = files.upload()
for fname in uploaded:
    os.rename(fname, '/content/network-traffic-transformer-lstm/dataset/raw/darknet2020.parquet')

Upload cicdarknet2020.parquet


Saving cicdarknet2020.parquet to cicdarknet2020.parquet


Step 4: Upload our second dataset (ISCXTor2016)
Our teacher asked us to compare on at least 2 datasets, so we use this second one too. It comes from the same research lab as the first dataset.

In [4]:
print("Upload Scenario-A-merged_5s.csv")
uploaded = files.upload()
for fname in uploaded:
    os.rename(fname, '/content/network-traffic-transformer-lstm/dataset/raw/iscxtor2016.csv')

Upload Scenario-A-merged_5s.csv


Saving Scenario-A-merged_5s.csv to Scenario-A-merged_5s.csv


Step 5: Clean and prepare both datasets
This step removes duplicate rows, removes rows with missing or broken values, removes columns that don't help (like IP addresses), turns the Tor/Non-Tor labels into 0s and 1s, and splits the data into training, validation, and testing parts.

In [5]:
%cd /content/network-traffic-transformer-lstm/src
!python preprocessing.py --dataset darknet2020
!python preprocessing.py --dataset iscxtor2016

/content/network-traffic-transformer-lstm/src
[main] Found dataset file: ../dataset/raw/darknet2020.parquet

===== Preprocessing darknet2020 =====
[load] Detected Parquet file, reading with pd.read_parquet()
[label] Using 'Label' as the raw label column
[label] Dropped 37138 rows that were neither Tor nor Non-Tor (e.g. VPN/NonVPN rows not relevant to this binary task)
[clean] Starting shape: (65983, 80)
[clean] Removed 0 duplicate rows
[clean] Removed 0 rows with missing/invalid values
[clean] Final shape: (65983, 80)
[features] Dropping 15 constant columns: ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Fwd Bytes/Bulk Avg', 'Fwd Packet/Bulk Avg', 'Fwd Bulk Rate Avg', 'Bwd Bytes/Bulk Avg', 'Subflow Bwd Packets', 'Active Mean', 'Active Std', 'Active Max', 'Active Min']
[features] Selected 62 numeric features
[label] Class mapping: {'NonTor': np.int64(0), 'Tor': np.int64(1)}
[label] Class balance: {np.int64(0): np.int64(64804), 

Step 6: Training and testing: Logistic Regression, SVM, LSTM only, CNN only, CNN + LSTM, and Transformer + LSTM on darknet2020 dataset

In [6]:
!python train.py --dataset darknet2020 --model logistic_regression
!python evaluate.py --dataset darknet2020 --model logistic_regression

!python train.py --dataset darknet2020 --model svm
!python evaluate.py --dataset darknet2020 --model svm

!python train.py --dataset darknet2020 --model lstm_only --epochs 40
!python evaluate.py --dataset darknet2020 --model lstm_only

!python train.py --dataset darknet2020 --model cnn_only --epochs 40
!python evaluate.py --dataset darknet2020 --model cnn_only

!python train.py --dataset darknet2020 --model cnn_lstm --epochs 40
!python evaluate.py --dataset darknet2020 --model cnn_lstm

!python train.py --dataset darknet2020 --model transformer_lstm --epochs 40
!python evaluate.py --dataset darknet2020 --model transformer_lstm --primary

2026-06-27 14:07:15.492282: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-27 14:07:15.564375: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[train] Training took 1.0 seconds
[train] Saved model to ../results/models/darknet2020_logistic_regression.joblib
2026-06-27 14:07:24.852759: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-27 14:07:24.925740: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow wit

Step 7: Training and testing: Logistic Regression, SVM, LSTM only, CNN only, CNN + LSTM, and Transformer + LSTM on iscxtor2016 dataset

In [7]:
!python train.py --dataset iscxtor2016 --model logistic_regression
!python evaluate.py --dataset iscxtor2016 --model logistic_regression

!python train.py --dataset iscxtor2016 --model svm
!python evaluate.py --dataset iscxtor2016 --model svm

!python train.py --dataset iscxtor2016 --model lstm_only --epochs 40
!python evaluate.py --dataset iscxtor2016 --model lstm_only

!python train.py --dataset iscxtor2016 --model cnn_only --epochs 40
!python evaluate.py --dataset iscxtor2016 --model cnn_only

!python train.py --dataset iscxtor2016 --model cnn_lstm --epochs 40
!python evaluate.py --dataset iscxtor2016 --model cnn_lstm

!python train.py --dataset iscxtor2016 --model transformer_lstm --epochs 40
!python evaluate.py --dataset iscxtor2016 --model transformer_lstm

2026-06-27 14:12:23.968004: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-27 14:12:24.041328: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[train] Training took 1.4 seconds
[train] Saved model to ../results/models/iscxtor2016_logistic_regression.joblib
2026-06-27 14:12:32.955911: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-27 14:12:33.088681: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow wit

Step 7: Look at all our results together and Look at our model's confusion matrix and graphs